# Proby Parquet Merge

This notebook vertically merges the existing normalized Proby parquet files into one parquet file.

It does not add `BatchCode`, because the merged dataset is intended to remove the batch concept for later MyLabData changes.

The merge is streaming and row-group based, so it does not load all Proby data into memory at once.

In [6]:
from pathlib import Path
import math

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# Source directory containing BatchE/BacthG Proby parquet files.
PROBY_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby")

# Merged output. This file is intentionally excluded from future scans.
OUTPUT_PATH = PROBY_DIR / "Proby_All.parquet"

# Set to True only when you intentionally want to replace an existing output file.
OVERWRITE_OUTPUT = False

PROBY_COLUMNS = [
    "SMILES",
    "abs",
    "emi",
    "plqy",
    "e",
    "log10e",
    "lifetime",
    "abs_fwhm_cm",
    "emi_fwhm_cm",
    "abs_fwhm_nm",
    "emi_fwhm_nm",
    "Solvent",
]

TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("abs", pa.float64()),
    pa.field("emi", pa.float64()),
    pa.field("plqy", pa.float64()),
    pa.field("e", pa.float64()),
    pa.field("log10e", pa.float64()),
    pa.field("lifetime", pa.float64()),
    pa.field("abs_fwhm_cm", pa.float64()),
    pa.field("emi_fwhm_cm", pa.float64()),
    pa.field("abs_fwhm_nm", pa.float64()),
    pa.field("emi_fwhm_nm", pa.float64()),
    pa.field("Solvent", pa.large_string()),
])

print(f"Proby directory: {PROBY_DIR}")
print(f"Output path:     {OUTPUT_PATH}")

Proby directory: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby
Output path:     C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All.parquet


## Convert Batch Folders To Batch Parquets

This standalone cell converts each `BatchE0XX` folder of Proby Excel slice files into one batch-level parquet file. The output schema matches the Proby merged table: `SMILES`, prediction columns, and `Solvent`.


In [ ]:
# Standalone cell: convert each Proby Batch folder into one parquet file.
# You can run this cell without running any previous cell in this notebook.

from pathlib import Path
import math
import re

import pyarrow as pa
import pyarrow.parquet as pq
from openpyxl import load_workbook
from tqdm.notebook import tqdm

# Path settings: edit these absolute paths as needed.
RUN_BATCH_FOLDER_TO_PARQUET = True

PROBY_ORIGIN_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\Proby")
PROBY_BATCH_PARQUET_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby")

# Process all Batch* folders by default. You can restrict this, for example: ["BatchE018", "BatchE019"]
BATCH_FOLDER_NAMES = BatchE001

# Smaller chunks use less memory. Increase only if your machine handles it comfortably.
EXCEL_ROWS_PER_ARROW_BATCH = 100000

# Set to True only when you intentionally want to replace existing batch parquet files.
OVERWRITE_BATCH_PARQUETS = False

PROBY_EXCEL_TO_PARQUET_COLUMNS = {
    "smiles": "SMILES",
    "abs": "abs",
    "emi": "emi",
    "plqy": "plqy",
    "e": "e",
    "log10e": "log10e",
    "lifetime": "lifetime",
    "abs fwhm (cm-1)": "abs_fwhm_cm",
    "emi fwhm (cm-1)": "emi_fwhm_cm",
    "abs fwhm (nm)": "abs_fwhm_nm",
    "emi fwhm (nm)": "emi_fwhm_nm",
}

BATCH_PARQUET_COLUMNS = [
    "SMILES",
    "abs",
    "emi",
    "plqy",
    "e",
    "log10e",
    "lifetime",
    "abs_fwhm_cm",
    "emi_fwhm_cm",
    "abs_fwhm_nm",
    "emi_fwhm_nm",
    "Solvent",
]

BATCH_PARQUET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("abs", pa.float64()),
    pa.field("emi", pa.float64()),
    pa.field("plqy", pa.float64()),
    pa.field("e", pa.float64()),
    pa.field("log10e", pa.float64()),
    pa.field("lifetime", pa.float64()),
    pa.field("abs_fwhm_cm", pa.float64()),
    pa.field("emi_fwhm_cm", pa.float64()),
    pa.field("abs_fwhm_nm", pa.float64()),
    pa.field("emi_fwhm_nm", pa.float64()),
    pa.field("Solvent", pa.large_string()),
])


def solvent_from_sheet_name(sheet_name: str) -> str:
    # Sheet names look like "CC#N (5)". The trailing number is the solvent index.
    return re.sub(r"\s*\(\d+\)\s*$", "", sheet_name).strip()


def list_batch_folders(origin_dir: Path, names: list[str] | None = None) -> list[Path]:
    if not origin_dir.exists():
        raise FileNotFoundError(f"Origin directory does not exist: {origin_dir}")
    if names:
        folders = [origin_dir / name for name in names]
    else:
        folders = sorted(path for path in origin_dir.iterdir() if path.is_dir() and path.name.startswith("Batch"))
    missing = [path for path in folders if not path.exists() or not path.is_dir()]
    if missing:
        raise FileNotFoundError(f"Batch folders not found: {missing}")
    return folders


def empty_batch_accumulator() -> dict[str, list]:
    return {column: [] for column in BATCH_PARQUET_COLUMNS}


def append_excel_row(accumulator: dict[str, list], row_values: tuple, header_index: dict[str, int], solvent: str) -> None:
    smiles = row_values[header_index["smiles"]]
    if smiles is None or str(smiles).strip() == "":
        return

    accumulator["SMILES"].append(str(smiles).strip())
    for source_column, output_column in PROBY_EXCEL_TO_PARQUET_COLUMNS.items():
        if output_column == "SMILES":
            continue
        value = row_values[header_index[source_column]]
        accumulator[output_column].append(None if value is None else float(value))
    accumulator["Solvent"].append(solvent)


def accumulator_to_table(accumulator: dict[str, list]) -> pa.Table:
    arrays = [pa.array(accumulator[column], type=BATCH_PARQUET_SCHEMA.field(column).type) for column in BATCH_PARQUET_COLUMNS]
    return pa.Table.from_arrays(arrays, schema=BATCH_PARQUET_SCHEMA)


def write_accumulator(writer: pq.ParquetWriter, accumulator: dict[str, list]) -> int:
    row_count = len(accumulator["SMILES"])
    if row_count == 0:
        return 0
    writer.write_table(accumulator_to_table(accumulator))
    return row_count


def convert_one_proby_batch_folder(batch_folder: Path, output_dir: Path, overwrite: bool = False, rows_per_batch: int = 10_000) -> dict:
    if rows_per_batch < 1:
        raise ValueError("EXCEL_ROWS_PER_ARROW_BATCH must be at least 1.")

    xlsx_files = sorted(batch_folder.glob("*.xlsx"))
    if not xlsx_files:
        raise FileNotFoundError(f"No xlsx files found in {batch_folder}")

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{batch_folder.name}.parquet"
    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")

    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_BATCH_PARQUETS = True to replace it.")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_output_path}")

    writer = pq.ParquetWriter(
        temp_output_path,
        BATCH_PARQUET_SCHEMA,
        compression="snappy",
        use_dictionary=["SMILES", "Solvent"],
    )
    rows_written = 0
    sheets_written = 0

    try:
        file_progress = tqdm(xlsx_files, desc=batch_folder.name, unit="file", dynamic_ncols=True, leave=False)
        for xlsx_path in file_progress:
            file_progress.set_postfix_str(xlsx_path.name)
            workbook = load_workbook(xlsx_path, read_only=True, data_only=True)
            try:
                for worksheet in tqdm(workbook.worksheets, desc=xlsx_path.stem, unit="sheet", dynamic_ncols=True, leave=False):
                    rows_iter = worksheet.iter_rows(values_only=True)
                    try:
                        header = next(rows_iter)
                    except StopIteration:
                        continue

                    normalized_header = [str(value).strip() if value is not None else "" for value in header]
                    header_index = {name: idx for idx, name in enumerate(normalized_header)}
                    missing = [column for column in PROBY_EXCEL_TO_PARQUET_COLUMNS if column not in header_index]
                    if missing:
                        raise ValueError(f"{xlsx_path.name} / {worksheet.title} is missing columns: {missing}")

                    solvent = solvent_from_sheet_name(worksheet.title)
                    accumulator = empty_batch_accumulator()
                    for row_values in rows_iter:
                        append_excel_row(accumulator, row_values, header_index, solvent)
                        if len(accumulator["SMILES"]) >= rows_per_batch:
                            rows_written += write_accumulator(writer, accumulator)
                            accumulator = empty_batch_accumulator()

                    rows_written += write_accumulator(writer, accumulator)
                    sheets_written += 1
            finally:
                workbook.close()
    finally:
        writer.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError(f"No rows were written for {batch_folder}")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)

    return {
        "batch": batch_folder.name,
        "output_path": str(output_path),
        "xlsx_files": len(xlsx_files),
        "sheets_written": sheets_written,
        "rows_written": rows_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


if RUN_BATCH_FOLDER_TO_PARQUET:
    batch_folders = list_batch_folders(PROBY_ORIGIN_DIR, BATCH_FOLDER_NAMES)
    print(f"Batch folders: {len(batch_folders)}")
    print(f"Output directory: {PROBY_BATCH_PARQUET_DIR}")

    batch_results = []
    for batch_folder in tqdm(batch_folders, desc="Convert Proby batches", unit="batch", dynamic_ncols=True):
        result = convert_one_proby_batch_folder(
            batch_folder=batch_folder,
            output_dir=PROBY_BATCH_PARQUET_DIR,
            overwrite=OVERWRITE_BATCH_PARQUETS,
            rows_per_batch=EXCEL_ROWS_PER_ARROW_BATCH,
        )
        batch_results.append(result)
        print(result)

    print("Finished converting batch folders.")
else:
    print("Set RUN_BATCH_FOLDER_TO_PARQUET = True to convert Proby Batch folders into batch parquet files.")


## Scan Inputs

This cell finds normalized batch parquet files in `PROBY_DIR`, excluding the merged output and temporary output files.

In [7]:
def discover_input_files(proby_dir: Path, output_path: Path) -> list[Path]:
    files = []
    for path in sorted(proby_dir.glob("*.parquet")):
        if path.resolve() == output_path.resolve():
            continue
        if path.name.endswith(".tmp.parquet"):
            continue
        if path.stem.startswith("Proby_All"):
            continue
        files.append(path)
    return files


input_files = discover_input_files(PROBY_DIR, OUTPUT_PATH)
if not input_files:
    raise FileNotFoundError(f"No input parquet files found in {PROBY_DIR}")

total_rows = 0
total_bytes = 0
for path in input_files:
    metadata = pq.ParquetFile(path).metadata
    total_rows += metadata.num_rows
    total_bytes += path.stat().st_size
    print(f"{path.name:20s} rows={metadata.num_rows:>12,} row_groups={metadata.num_row_groups:>5,} size={path.stat().st_size / 1024**2:>9.2f} MB")

print("-" * 80)
print(f"Input files: {len(input_files)}")
print(f"Total rows:  {total_rows:,}")
print(f"Total size:  {total_bytes / 1024**3:.2f} GiB")

BatchE001.parquet    rows=  66,011,478 row_groups=   63 size=  2550.37 MB
BatchE002.parquet    rows= 110,608,836 row_groups=1,144 size=  3080.73 MB
BatchE003.parquet    rows=  12,999,922 row_groups=1,300 size=   471.00 MB
BatchE004.parquet    rows=  12,999,896 row_groups=  130 size=   509.41 MB
BatchE005.parquet    rows=  12,999,896 row_groups=1,300 size=   507.55 MB
BatchE006.parquet    rows=   1,642,472 row_groups=  182 size=    70.70 MB
BatchE007.parquet    rows=  16,117,244 row_groups=1,612 size=   433.45 MB
BatchE008.parquet    rows=   8,039,824 row_groups=  806 size=   252.07 MB
BatchE009.parquet    rows=   6,257,966 row_groups=  650 size=   164.78 MB
BatchE010.parquet    rows=  18,719,870 row_groups=1,872 size=   455.80 MB
BatchE011.parquet    rows=  18,719,688 row_groups=1,872 size=   455.50 MB
BatchE012.parquet    rows=   8,559,694 row_groups=  858 size=   235.48 MB
BatchE013.parquet    rows=   1,900,678 row_groups=  208 size=    40.22 MB
BatchE014.parquet    rows=  18,199,818

## Validate Schemas

The existing files may use either Arrow `string` or `large_string` for `SMILES` and `Solvent`. The merge normalizes both to `large_string`.

In [8]:
def validate_input_schema(path: Path) -> None:
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in PROBY_COLUMNS if column not in schema.names]
    extra = [name for name in schema.names if name not in PROBY_COLUMNS]
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")
    if extra:
        raise ValueError(f"{path.name} has unexpected columns: {extra}")


for path in input_files:
    validate_input_schema(path)

print(f"Validated {len(input_files)} parquet file(s).")
print(TARGET_SCHEMA)

Validated 28 parquet file(s).
SMILES: large_string
abs: double
emi: double
plqy: double
e: double
log10e: double
lifetime: double
abs_fwhm_cm: double
emi_fwhm_cm: double
abs_fwhm_nm: double
emi_fwhm_nm: double
Solvent: large_string


## Merge Function

This function reads each source parquet row group, casts columns to a single target schema, and appends it to the merged output.

In [9]:
def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def merge_proby_parquets(input_paths: list[Path], output_path: Path, overwrite: bool = False) -> dict:
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_OUTPUT = True to replace it.")

    temp_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_path.exists():
        if overwrite:
            temp_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_path}")

    writer = None
    rows_written = 0
    row_groups_written = 0

    try:
        writer = pq.ParquetWriter(
            temp_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES", "Solvent"],
        )

        for file_index, path in enumerate(input_paths, start=1):
            parquet_file = pq.ParquetFile(path)
            print(f"[{file_index}/{len(input_paths)}] {path.name}: {parquet_file.metadata.num_rows:,} rows")

            for row_group_index in range(parquet_file.metadata.num_row_groups):
                table = parquet_file.read_row_group(row_group_index, columns=PROBY_COLUMNS)
                table = align_table(table)
                writer.write_table(table)
                rows_written += table.num_rows
                row_groups_written += 1

    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; merge aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_path.replace(output_path)

    return {
        "output_path": str(output_path),
        "rows_written": rows_written,
        "row_groups_written": row_groups_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }

## Run Merge

Set `RUN_MERGE = True` when you are ready. The output will be `Proby_All.parquet` in the Proby directory.

In [10]:
RUN_MERGE = False

if RUN_MERGE:
    result = merge_proby_parquets(input_files, OUTPUT_PATH, overwrite=OVERWRITE_OUTPUT)
    print("Merge complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_MERGE = True to create the merged parquet file.")

Dry run only. Set RUN_MERGE = True to create the merged parquet file.


## Deduplicate Merged Output

Run this cell after a merged parquet file has been created. Set `DEDUP_INPUT_PATH` and `DEDUP_OUTPUT_PATH` to absolute parquet paths, then it reads the input file in small batches, keeps the first row for each configured deduplication key, and writes a separate deduplicated parquet file.

This version is designed to minimize memory use. It stores the global set of seen keys in a temporary SQLite database on disk, instead of keeping all keys in Python memory. It is slower than in-memory deduplication, but much safer for very large parquet files.

Choose any one or more column names in `DEDUP_COLUMNS`. Examples: `["SMILES"]`, `["SMILES", "Solvent"]`, or `["SMILES", "Solvent", "abs", "emi"]`.

In [12]:
# ?? Dedup settings ??
RUN_DEDUP = True

# Use absolute paths here. The defaults point to the Proby merge output.
DEDUP_INPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup.parquet")
DEDUP_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X.parquet")

# Pick any one or more columns from PROBY_COLUMNS.
DEDUP_COLUMNS = ["SMILES"]

# Smaller batches use less memory. Increase only if your machine handles it comfortably.
DEDUP_BATCH_SIZE = 10000

# SQLite stores the global seen-key index on disk, keeping Python memory low.
# Temporary SQLite key database. Default: next to the output parquet.
DEDUP_SQLITE_PATH = DEDUP_OUTPUT_PATH.with_name(DEDUP_OUTPUT_PATH.stem + "_seen.sqlite")

# Streaming dedup keeps the first row encountered for each dedup key.
DEDUP_KEEP = "first"

# Set to True only when you intentionally want to replace the dedup output file.
OVERWRITE_DEDUP_OUTPUT = False

# Remove the temporary SQLite key database after successful deduplication.
CLEAN_DEDUP_TEMP = True


def validate_dedup_settings(columns: list[str], keep: str, batch_size: int) -> None:
    if keep != "first":
        raise ValueError("Only DEDUP_KEEP = 'first' is supported for streaming deduplication.")
    if not columns:
        raise ValueError("DEDUP_COLUMNS must contain at least one column.")
    missing = [column for column in columns if column not in PROBY_COLUMNS]
    if missing:
        raise ValueError(f"DEDUP_COLUMNS contains unknown columns: {missing}")
    if batch_size < 1:
        raise ValueError("DEDUP_BATCH_SIZE must be at least 1.")


def _dedup_value(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def _dedup_key(value_tuple: tuple) -> str:
    # repr is deterministic for the scalar strings/floats/None used here and avoids JSON overhead.
    return repr(tuple(_dedup_value(value) for value in value_tuple))


def make_key_rows(table: pa.Table, dedup_columns: list[str]) -> list[tuple[int, str]]:
    key_columns = [table[column].to_pylist() for column in dedup_columns]
    rows = []
    for row_index, values in enumerate(zip(*key_columns)):
        rows.append((row_index, _dedup_key(values)))
    return rows


def setup_seen_key_db(db_path: Path, overwrite: bool) -> "sqlite3.Connection":
    import sqlite3

    if db_path.exists():
        if overwrite:
            db_path.unlink()
        else:
            raise FileExistsError(f"Temporary SQLite key DB already exists: {db_path}")

    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("PRAGMA temp_store=FILE")
    conn.execute("PRAGMA cache_size=-200000")  # around 200 MB SQLite page cache cap
    conn.execute("CREATE TABLE seen_keys (key TEXT PRIMARY KEY)")
    conn.execute("CREATE TEMP TABLE batch_keys (pos INTEGER NOT NULL, key TEXT NOT NULL)")
    conn.execute("CREATE INDEX batch_keys_key_pos_idx ON batch_keys(key, pos)")
    conn.commit()
    return conn


def find_new_positions(conn, key_rows: list[tuple[int, str]]) -> list[int]:
    conn.execute("DELETE FROM batch_keys")
    conn.executemany("INSERT INTO batch_keys(pos, key) VALUES (?, ?)", key_rows)

    # Keep the first row per key within this batch, but only if the key has not appeared before.
    keep_rows = conn.execute("""
        SELECT MIN(b.pos) AS pos, b.key
        FROM batch_keys b
        LEFT JOIN seen_keys s ON s.key = b.key
        WHERE s.key IS NULL
        GROUP BY b.key
    """).fetchall()

    if keep_rows:
        conn.executemany("INSERT OR IGNORE INTO seen_keys(key) VALUES (?)", [(key,) for _, key in keep_rows])
    conn.commit()
    return [pos for pos, _ in keep_rows]


def filter_positions(table: pa.Table, keep_positions: list[int]) -> pa.Table:
    if not keep_positions:
        return table.slice(0, 0)
    keep_positions.sort()
    return table.take(pa.array(keep_positions, type=pa.int64()))


def deduplicate_parquet_sqlite(
    input_path: Path,
    output_path: Path,
    dedup_columns: list[str],
    batch_size: int,
    sqlite_path: Path,
    overwrite: bool = False,
    clean_temp: bool = True,
) -> dict:
    validate_dedup_settings(dedup_columns, DEDUP_KEEP, batch_size)
    if not input_path.exists():
        raise FileNotFoundError(f"Dedup input does not exist: {input_path}")
    if input_path.resolve() == output_path.resolve():
        raise ValueError("DEDUP_OUTPUT_PATH must be different from DEDUP_INPUT_PATH.")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Dedup output already exists: {output_path}. Set OVERWRITE_DEDUP_OUTPUT = True to replace it.")

    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary dedup output already exists: {temp_output_path}")

    conn = setup_seen_key_db(sqlite_path, overwrite=overwrite)
    parquet_file = pq.ParquetFile(input_path)
    writer = None
    rows_read = 0
    rows_written = 0
    duplicate_rows_skipped = 0
    batches_written = 0

    total_rows = parquet_file.metadata.num_rows
    progress = tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=PROBY_COLUMNS),
        total=math.ceil(total_rows / batch_size),
        desc="Deduplicate parquet",
        unit="batch",
        dynamic_ncols=True,
    )

    try:
        writer = pq.ParquetWriter(
            temp_output_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES", "Solvent"],
        )

        for record_batch in progress:
            table = pa.Table.from_batches([record_batch])
            rows_read += table.num_rows
            table = align_table(table)

            key_rows = make_key_rows(table, dedup_columns)
            keep_positions = find_new_positions(conn, key_rows)
            duplicate_rows_skipped += table.num_rows - len(keep_positions)

            if keep_positions:
                filtered = filter_positions(table, keep_positions)
                writer.write_table(filtered)
                rows_written += filtered.num_rows
                batches_written += 1

            progress.set_postfix(
                read=f"{rows_read:,}",
                written=f"{rows_written:,}",
                skipped=f"{duplicate_rows_skipped:,}",
            )

    finally:
        progress.close()
        if writer is not None:
            writer.close()
        conn.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; dedup aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)

    if clean_temp:
        sqlite_path.unlink(missing_ok=True)

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dedup_columns": dedup_columns,
        "batch_size": batch_size,
        "sqlite_path": str(sqlite_path),
        "rows_read": rows_read,
        "rows_written": rows_written,
        "duplicate_rows_skipped": duplicate_rows_skipped,
        "batches_written": batches_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


if RUN_DEDUP:
    result = deduplicate_parquet_sqlite(
        DEDUP_INPUT_PATH,
        DEDUP_OUTPUT_PATH,
        DEDUP_COLUMNS,
        DEDUP_BATCH_SIZE,
        DEDUP_SQLITE_PATH,
        overwrite=OVERWRITE_DEDUP_OUTPUT,
        clean_temp=CLEAN_DEDUP_TEMP,
    )
    print("Dedup complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_DEDUP = True to create the deduplicated parquet file.")
    print(f"Dedup input:    {DEDUP_INPUT_PATH}")
    print(f"Dedup output:   {DEDUP_OUTPUT_PATH}")
    print(f"Dedup columns:  {DEDUP_COLUMNS}")
    print(f"Batch size:     {DEDUP_BATCH_SIZE:,}")
    print(f"SQLite DB:      {DEDUP_SQLITE_PATH}")

Deduplicate parquet:   0%|          | 0/27603 [00:00<?, ?batch/s]

Dedup complete.
input_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup.parquet
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X.parquet
dedup_columns: ['SMILES']
batch_size: 10000
sqlite_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X_seen.sqlite
rows_read: 276023150
rows_written: 10616275
duplicate_rows_skipped: 265406875
batches_written: 1742
size_gib: 0.30854695849120617


## Export SMILES With Missing Proby Predictions

This standalone cell can be run by itself. Set absolute input/output parquet paths, then it exports rows where `SMILES` is present and all Proby prediction value columns are missing.

Use this on a parquet that has already been deduplicated by `SMILES` if you want a unique SMILES-level missing-prediction list.

In [1]:
# Standalone cell: export SMILES rows with no Proby prediction values.
# You can run this cell without running any previous cell in this notebook.

from pathlib import Path
import math

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ?? Path settings: edit these absolute paths as needed ??
RUN_MISSING_PREDICTION_EXPORT = True

MISSING_INPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X.parquet")
MISSING_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X_SMILES.parquet")

# Smaller batches use less memory. Increase only if your machine handles it comfortably.
MISSING_BATCH_SIZE = 50_000

# Set to True only when you intentionally want to replace an existing output file.
OVERWRITE_MISSING_OUTPUT = False

SMILES_COLUMN = "SMILES"
PROBY_RESULT_COLUMNS = [
    "abs", "emi", "plqy", "e", "log10e", "lifetime",
    "abs_fwhm_cm", "emi_fwhm_cm", "abs_fwhm_nm", "emi_fwhm_nm",
]
MISSING_READ_COLUMNS = [SMILES_COLUMN] + PROBY_RESULT_COLUMNS

# Output only SMILES by default. Set to MISSING_READ_COLUMNS if you want to keep the empty metric columns too.
MISSING_OUTPUT_COLUMNS = [SMILES_COLUMN]


def validate_missing_prediction_input(path: Path) -> pa.Schema:
    if not path.exists():
        raise FileNotFoundError(f"Input parquet does not exist: {path}")
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in MISSING_READ_COLUMNS if column not in schema.names]
    if missing:
        raise ValueError(f"Input parquet is missing required columns: {missing}")
    return schema


def missing_prediction_mask(table: pa.Table) -> pa.Array:
    smiles = table[SMILES_COLUMN]
    mask = pc.and_(pc.is_valid(smiles), pc.not_equal(smiles, ""))
    for column in PROBY_RESULT_COLUMNS:
        mask = pc.and_(mask, pc.is_null(table[column], nan_is_null=True))
    return mask


def align_missing_output(table: pa.Table) -> pa.Table:
    arrays = []
    fields = []
    for column in MISSING_OUTPUT_COLUMNS:
        array = table[column]
        if column == SMILES_COLUMN and not pa.types.is_large_string(array.type):
            array = pc.cast(array, pa.large_string())
        arrays.append(array)
        fields.append(pa.field(column, array.type))
    return pa.Table.from_arrays(arrays, schema=pa.schema(fields))


def export_missing_predictions(
    input_path: Path,
    output_path: Path,
    batch_size: int = 50_000,
    overwrite: bool = False,
) -> dict:
    validate_missing_prediction_input(input_path)
    if input_path.resolve() == output_path.resolve():
        raise ValueError("MISSING_OUTPUT_PATH must be different from MISSING_INPUT_PATH.")
    if batch_size < 1:
        raise ValueError("MISSING_BATCH_SIZE must be at least 1.")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_MISSING_OUTPUT = True to replace it.")

    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_output_path}")

    parquet_file = pq.ParquetFile(input_path)
    writer = None
    rows_read = 0
    rows_written = 0
    batches_written = 0
    output_schema = None

    progress = tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=MISSING_READ_COLUMNS),
        total=math.ceil(parquet_file.metadata.num_rows / batch_size),
        desc="Find missing predictions",
        unit="batch",
        dynamic_ncols=True,
    )

    try:
        for record_batch in progress:
            table = pa.Table.from_batches([record_batch])
            rows_read += table.num_rows
            filtered = table.filter(missing_prediction_mask(table))
            if filtered.num_rows == 0:
                progress.set_postfix(read=f"{rows_read:,}", written=f"{rows_written:,}")
                continue

            output_table = align_missing_output(filtered)
            if writer is None:
                output_schema = output_table.schema
                writer = pq.ParquetWriter(
                    temp_output_path,
                    output_schema,
                    compression="snappy",
                    use_dictionary=[SMILES_COLUMN] if SMILES_COLUMN in MISSING_OUTPUT_COLUMNS else None,
                )

            writer.write_table(output_table)
            rows_written += output_table.num_rows
            batches_written += 1
            progress.set_postfix(read=f"{rows_read:,}", written=f"{rows_written:,}")

    finally:
        progress.close()
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError("No rows matched the missing-prediction filter; no output was written.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "rows_read": rows_read,
        "rows_written": rows_written,
        "batches_written": batches_written,
        "output_columns": MISSING_OUTPUT_COLUMNS,
        "size_mib": output_path.stat().st_size / 1024**2,
    }


if RUN_MISSING_PREDICTION_EXPORT:
    result = export_missing_predictions(
        MISSING_INPUT_PATH,
        MISSING_OUTPUT_PATH,
        batch_size=MISSING_BATCH_SIZE,
        overwrite=OVERWRITE_MISSING_OUTPUT,
    )
    print("Missing-prediction export complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_MISSING_PREDICTION_EXPORT = True to create the output parquet file.")
    print(f"Input:          {MISSING_INPUT_PATH}")
    print(f"Output:         {MISSING_OUTPUT_PATH}")
    print(f"Batch size:     {MISSING_BATCH_SIZE:,}")
    print(f"Output columns: {MISSING_OUTPUT_COLUMNS}")


Find missing predictions:   0%|          | 0/213 [00:00<?, ?batch/s]

Missing-prediction export complete.
input_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X.parquet
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\Proby_All_dedup_X_SMILES.parquet
rows_read: 10616275
rows_written: 9593774
batches_written: 213
output_columns: ['SMILES']
size_mib: 193.25087070465088


## Verify Output

Run this after the merge finishes.

In [ ]:
if OUTPUT_PATH.exists():
    merged = pq.ParquetFile(OUTPUT_PATH)
    print(f"Output:     {OUTPUT_PATH}")
    print(f"Rows:       {merged.metadata.num_rows:,}")
    print(f"Row groups: {merged.metadata.num_row_groups:,}")
    print(f"Size:       {OUTPUT_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(merged.schema_arrow)
else:
    print(f"Output does not exist yet: {OUTPUT_PATH}")